# Lejátszási listák szétválasztása

In [ ]:
import h5py
import numpy as np

H5_PATH = "../Dataset/spotify_dataset.h5"

with h5py.File(H5_PATH, "r") as hf:
    pids = hf["playlist_tracks/pid"][:]
    # Kigyűjtjük az egyedi lejátszási lista azonosítókat
    unique_pids = list(set(pids))

# 1. Megkeverjük őket
np.random.seed(42)
np.random.shuffle(unique_pids)

# 2. Kiszámoljuk a vágási pontokat (80% Train, 10% Val, 10% Test)
train_split = int(len(unique_pids) * 0.8)
val_split = int(len(unique_pids) * 0.9)

train_pids = unique_pids[:train_split]
val_pids = unique_pids[train_split:val_split]
test_pids = unique_pids[val_split:]

# 3. Kimentjük őket apró numpy fájlokba
np.save("train_pids.npy", train_pids)
np.save("val_pids.npy", val_pids)
np.save("test_pids.npy", test_pids)

print(f"Kész! Train: {len(train_pids)}, Val: {len(val_pids)}, Test: {len(test_pids)}")

Kész! Train: 800000, Val: 100000, Test: 100000


# Model tanítása

In [ ]:
import os
import pickle
import h5py
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input, GRU, Dense, Masking
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.sequence import pad_sequences
from gensim.models import Word2Vec
from collections import defaultdict
from tqdm import tqdm
import gc

# ==========================================
# --- ÚTVONALAK ÉS BEÁLLÍTÁSOK ---
# ==========================================
H5_PATH = "/content/spotify_dataset_compressed.h5"
W2V_PATH = "/content/song2vec.model"
CNN_PATH = "/content/spotify_cnn_model.keras"
GRU_SAVE_PATH = "/content/drive/MyDrive/spotify_gru4rec_hybrid.keras"

CNN_DICT_PATH = "/content/drive/MyDrive/cnn_vectors.pkl"
TRAIN_PIDS_PATH = "/content/train_pids.npy"
VAL_PIDS_PATH = "/content/val_pids.npy"

SEQ_LEN = 10
BATCH_SIZE = 64
EPOCHS = 10
EMBEDDING_DIM = 400

# ==========================================
# 1. & 2. & 3. CUSTOM RÉTEGEK, GENERÁTOR, MODELL
# ==========================================
class L2NormLayer(tf.keras.layers.Layer):
    def call(self, x): return tf.math.l2_normalize(x, axis=1)

def cosine_loss(y_true, y_pred):
    return 1.0 - tf.reduce_mean(tf.reduce_sum(tf.math.l2_normalize(y_true, axis=1) * y_pred, axis=1))

class HybridSequenceGenerator(tf.keras.utils.Sequence):
    def __init__(self, playlists, w2v_model, cnn_vectors, seq_len, batch_size):
        self.playlists = playlists # Csak egy hivatkozás az eredeti listákra, 0 Byte plusz RAM!
        self.w2v_model = w2v_model
        self.cnn_vectors = cnn_vectors
        self.seq_len = seq_len
        self.batch_size = batch_size

        samples_list = []

        print(f"\nTanító minták indexelése ({len(playlists)} listán)...")
        for p_idx, playlist in enumerate(tqdm(playlists, desc="Indexelés")):
            if len(playlist) < 3: continue

            for i in range(1, len(playlist) - 1):
                anchor_uri = playlist[i]
                target_uri = playlist[i+1]

                if anchor_uri in self.cnn_vectors and target_uri in self.w2v_model.wv:
                    # TRÜKK: A szövegek helyett csak két koordinátát (számot) mentünk el!
                    samples_list.append((p_idx, i))

        # TRÜKK 2: Python lista helyett C-típusú Numpy tömböt (int32) csinálunk belőle.
        # Ez garantálja, hogy több millió minta is csak pár Megabájt RAM-ot egyen.
        self.samples = np.array(samples_list, dtype=np.int32)
        print(f"Összesen {len(self.samples)} db hibrid tanító minta indexelve!")

        # Megkeverjük a sorokat (az indexeket)
        np.random.shuffle(self.samples)

    def __len__(self):
        return int(np.ceil(len(self.samples) / self.batch_size))

    def __getitem__(self, idx):
        # Kiveszünk egy batch-nyi koordinátát (p_idx = playlist index, anchor_idx = horgony index)
        batch_indices = self.samples[idx * self.batch_size : (idx + 1) * self.batch_size]
        X_batch, y_batch = [], []

        for p_idx, anchor_idx in batch_indices:
            # Röptében, csak most olvassuk ki a szövegeket a memóriából
            playlist = self.playlists[p_idx]
            target_uri = playlist[anchor_idx + 1]
            context_uris = playlist[max(0, anchor_idx + 1 - self.seq_len) : anchor_idx + 1]

            seq_vectors = []
            for i, uri in enumerate(context_uris):
                if i == len(context_uris) - 1:
                    seq_vectors.append(self.cnn_vectors[uri])
                else:
                    if uri in self.w2v_model.wv:
                        seq_vectors.append(self.w2v_model.wv[uri])
                    else:
                        seq_vectors.append(np.zeros(400)) # EMBEDDING_DIM helyett hardcodeolt 400

            X_batch.append(seq_vectors)
            y_batch.append(self.w2v_model.wv[target_uri])

        X_batch_padded = pad_sequences(X_batch, maxlen=self.seq_len, dtype='float32', padding='pre')
        return X_batch_padded, np.array(y_batch, dtype='float32')

def build_gru4rec_model(seq_len, embedding_dim):
    inputs = Input(shape=(seq_len, embedding_dim), name="sequence_input")
    x = Masking(mask_value=0.0)(inputs)
    x = GRU(512, activation='tanh', dropout=0.2, recurrent_dropout=0.2, return_sequences=False)(x)
    
    output_raw = Dense(embedding_dim, activation='linear', name='word2vec_output')(x)
    outputs = L2NormLayer(name='l2_norm')(output_raw)

    model = Model(inputs=inputs, outputs=outputs)

    model.compile(optimizer='adam', loss=cosine_loss)
    return model

# ==========================================
# 4. FŐ CIKLUS
# ==========================================
def main():
    print("1. Modellek betöltése (Memória-optimalizált módban)...")
    # TRÜKK 1: Memory mapping a W2V-nek! Nem tölti be az egészet a RAM-ba.
    w2v_model = Word2Vec.load(W2V_PATH, mmap='r')
    cnn_model = tf.keras.models.load_model(
        CNN_PATH, custom_objects={'cosine_loss': cosine_loss, 'L2NormLayer': L2NormLayer}
    )

    with h5py.File(H5_PATH, "r") as hf:
        # Itt is már csak akkor olvassuk be a memóriába, ha kell
        all_uris = hf["ml/track_uri"][:]
        has_spec = np.any(hf["spectrograms/mel"][:], axis=(1, 2))

    # --- CNN VEKTOROK KINYERÉSE ---
    print("\n2. OFFLINE FEATURE EXTRACTION (CNN Vektorok)...")
    if os.path.exists(CNN_DICT_PATH):
        print("✅ Megtaláltam az elmentett CNN vektorokat a Drive-on! Betöltés...")
        with open(CNN_DICT_PATH, 'rb') as f:
            cnn_vectors_dict = pickle.load(f)
    else:
        print("⚠️ Nem találtam mentett vektorokat. CNN Inferálás indul...")
        cnn_vectors_dict = {}
        with h5py.File(H5_PATH, "r") as hf:
            spec_indices = [i for i, h in enumerate(has_spec) if h]
            for idx in tqdm(spec_indices, desc="CNN Inferálás"):
                uri = (all_uris[idx].decode('utf-8') if isinstance(all_uris[idx], bytes) else all_uris[idx])

                m = np.expand_dims(np.expand_dims(hf["spectrograms/mel"][idx], axis=0), axis=-1).astype(np.float32)
                c = np.expand_dims(np.expand_dims(hf["spectrograms/chroma"][idx], axis=0), axis=-1).astype(np.float32)
                t = np.expand_dims(np.expand_dims(hf["spectrograms/tempogram"][idx], axis=0), axis=-1).astype(np.float32)

                vec = cnn_model.predict([m, c, t], verbose=0)[0]
                cnn_vectors_dict[uri] = vec

        with open(CNN_DICT_PATH, 'wb') as f:
            pickle.dump(cnn_vectors_dict, f)

    # Létrehozunk egy halmazt a gyors kereséshez
    cnn_uri_set = set(cnn_vectors_dict.keys())

    # Memória takarítás az Inferálás után
    del all_uris
    gc.collect()

    # --- ADATOK BETÖLTÉSE ÉS SZŰRÉSE ---
    print("\n3. Lejátszási Listák szűrése és felépítése (Kétmenetes módszer)...")
    playlist_map = defaultdict(list)
    valid_pids = set()

    with h5py.File(H5_PATH, "r") as hf:
        num_records = hf["playlist_tracks/pid"].shape[0]
        chunk_size = 2_000_000 # 2 millió soronként haladunk (így a RAM fellélegzik)

        print("   -> 1. Menet: Értékes listák (CNN dalokat tartalmazók) megkeresése...")
        for start in tqdm(range(0, num_records, chunk_size), desc="1. Menet"):
            end = min(start + chunk_size, num_records)
            pids_chunk = hf["playlist_tracks/pid"][start:end]
            uris_chunk = hf["playlist_tracks/track_uri"][start:end]

            for i in range(len(pids_chunk)):
                uri = uris_chunk[i].decode('utf-8') if isinstance(uris_chunk[i], bytes) else uris_chunk[i]
                if uri in cnn_uri_set:
                    valid_pids.add(pids_chunk[i])

        print(f"   -> Találtam {len(valid_pids)} db értékes listát! (A többit ignoráljuk)")

        print("   -> 2. Menet: Csak az értékes listák betöltése a memóriába...")
        for start in tqdm(range(0, num_records, chunk_size), desc="2. Menet"):
            end = min(start + chunk_size, num_records)
            pids_chunk = hf["playlist_tracks/pid"][start:end]
            uris_chunk = hf["playlist_tracks/track_uri"][start:end]

            for i in range(len(pids_chunk)):
                pid = pids_chunk[i]
                # Csak akkor csinálunk bármit is, ha a PID értékes!
                if pid in valid_pids:
                    uri = uris_chunk[i].decode('utf-8') if isinstance(uris_chunk[i], bytes) else uris_chunk[i]
                    playlist_map[pid].append(uri)

    print(f"✅ Kész! A RAM-ban mindössze {len(playlist_map)} db lejátszási lista van, és nem omlottunk össze!")
    gc.collect()

    # --- LISTÁK SZÉTVÁLASZTÁSA ---
    print("\n4. Tanító és Validációs listák betöltése...")
    train_pids = np.load(TRAIN_PIDS_PATH)
    val_pids = np.load(VAL_PIDS_PATH)

    train_playlists = [playlist_map[pid] for pid in train_pids if pid in playlist_map]
    val_playlists = [playlist_map[pid] for pid in val_pids if pid in playlist_map]

    print("\n5. GRU4Rec Modell inicializálása...")
    gru_model = build_gru4rec_model(SEQ_LEN, EMBEDDING_DIM)

    print("\n6. Adatgenerátorok indítása...")
    train_gen = HybridSequenceGenerator(train_playlists, w2v_model, cnn_vectors_dict, SEQ_LEN, BATCH_SIZE)
    val_gen = HybridSequenceGenerator(val_playlists, w2v_model, cnn_vectors_dict, SEQ_LEN, BATCH_SIZE)

    print("\n7. TANÍTÁS INDÍTÁSA 🚀...")
    checkpoint = tf.keras.callbacks.ModelCheckpoint(
        filepath=GRU_SAVE_PATH,
        save_best_only=True,
        monitor='val_loss',
        mode='min',
        verbose=1
    )

    # Kiszámoljuk, mennyi 10.000 lépés
    STEPS_PER_EPOCH = 10000
    VAL_STEPS = 2000

    history = gru_model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=EPOCHS * 40, # Megnöveljük az epochok számát, mert "rövidebbek" lettek
        steps_per_epoch=STEPS_PER_EPOCH, # <--- EZ A TRÜKK!
        validation_steps=VAL_STEPS,      # <--- EZ IS!
        callbacks=[checkpoint]
    )

    print(f"\n✅ Tanítás befejezve! A legjobb modell elmentve ide: {GRU_SAVE_PATH}")

if __name__ == "__main__":
    main()

1. Modellek betöltése (Memória-optimalizált módban)...

2. OFFLINE FEATURE EXTRACTION (CNN Vektorok)...
✅ Megtaláltam az elmentett CNN vektorokat a Drive-on! Betöltés...

3. Lejátszási Listák szűrése és felépítése (Kétmenetes módszer)...
   -> 1. Menet: Értékes listák (CNN dalokat tartalmazók) megkeresése...


1. Menet: 100%|██████████| 34/34 [01:33<00:00,  2.74s/it]


   -> Találtam 971938 db értékes listát! (A többit ignoráljuk)
   -> 2. Menet: Csak az értékes listák betöltése a memóriába...


2. Menet: 100%|██████████| 34/34 [01:43<00:00,  3.04s/it]


✅ Kész! A RAM-ban mindössze 971938 db lejátszási lista van, és nem omlottunk össze!

4. Tanító és Validációs listák betöltése...

5. GRU4Rec Modell inicializálása...

6. Adatgenerátorok indítása...

Tanító minták indexelése (777616 listán)...


...

Összesen 25733259 db hibrid tanító minta indexelve!

Tanító minták indexelése (97151 listán)...


Indexelés: 100%|██████████| 97151/97151 [00:04<00:00, 24131.62it/s]


Összesen 3216212 db hibrid tanító minta indexelve!

7. TANÍTÁS INDÍTÁSA 🚀...


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/400
10000/10000 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - loss: -9880.1139
Epoch 1: val_loss improved from None to -39336.75391, saving model to /content/drive/MyDrive/spotify_gru4rec_hybrid.keras

Epoch 1: finished saving model to /content/drive/MyDrive/spotify_gru4rec_hybrid.keras
10000/10000 ━━━━━━━━━━━━━━━━━━━━ 488s 48ms/step - loss: -19719.5117 - val_loss: -39336.7539
Epoch 2/400
10000/10000 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - loss: -49228.2534
Epoch 2: val_loss improved from -39336.75391 to -78614.94531, saving model to /content/drive/MyDrive/spotify_gru4rec_hybrid.keras

Epoch 2: finished saving model to /content/drive/MyDrive/spotify_gru4rec_hybrid.keras
10000/10000 ━━━━━━━━━━━━━━━━━━━━ 468s 47ms/step - loss: -59050.7148 - val_loss: -78614.9453
Epoch 3/400
10000/10000 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - loss: -88527.0986
Epoch 3: val_loss improved from -78614.94531 to -117895.09375, saving model to /content/drive/MyDrive/spotify_gru4rec_hybrid.keras

Epoch 3: finished saving 

In [6]:
import os
import pickle
import h5py
import numpy as np
import tensorflow as tf
from gensim.models import Word2Vec
from collections import defaultdict
from tqdm import tqdm
from tensorflow.keras.preprocessing.sequence import pad_sequences

# ==========================================
# --- ÚTVONALAK ÉS BEÁLLÍTÁSOK ---
# ==========================================
H5_PATH = "../Dataset/spotify_dataset_compressed.h5"
W2V_PATH = "../Models/song2vec.model"
GRU_MODEL_PATH = "../Models/spotify_gru4rec_hybrid.keras"
CNN_DICT_PATH = "../Models/cnn_vectors.pkl" 
TEST_PIDS_PATH = "../test_pids.npy"

K_VALUE = 20 
NUM_TEST_PLAYLISTS = 1000 
SEQ_LEN = 10 

# ==========================================
# 1. CUSTOM RÉTEGEK ÉS METRIKÁK
# ==========================================
class L2NormLayer(tf.keras.layers.Layer):
    def call(self, x): return tf.math.l2_normalize(x, axis=1)

def cosine_loss(y_true, y_pred):
    return 1.0 - tf.reduce_mean(tf.reduce_sum(tf.math.l2_normalize(y_true, axis=1) * y_pred, axis=1))

def calculate_ranking_metrics(recommended_uris, relevant_uris, K):
    """HR, NDCG és AP számítása a Top-K ajánlások alapján."""
    top_k = recommended_uris[:K]
    hits = [1 if uri in relevant_uris else 0 for uri in top_k]
    
    # Hit Rate
    hr = 1 if sum(hits) > 0 else 0
    
    # Average Precision (MAP-hez)
    precs = []
    num_hits = 0
    for i, hit in enumerate(hits):
        if hit:
            num_hits += 1
            precs.append(num_hits / (i + 1))
    ap = np.mean(precs) if precs else 0
    
    # NDCG
    dcg = 0
    for i, hit in enumerate(hits):
        if hit:
            dcg += 1 / np.log2(i + 2)
    
    idcg = 0
    for i in range(min(len(relevant_uris), K)):
        idcg += 1 / np.log2(i + 2)
    
    ndcg = dcg / idcg if idcg > 0 else 0
    
    return hr, ndcg, ap

# ==========================================
# 2. KIÉRTÉKELŐ FŐ CIKLUS
# ==========================================
def main():
    print("1. Modellek és CNN szótár betöltése...")
    # Word2Vec betöltése mmap módban (RAM spórolás)
    w2v_model = Word2Vec.load(W2V_PATH, mmap='r')
    
    # GRU Hibrid modell betöltése a custom objektumokkal
    gru_model = tf.keras.models.load_model(
        GRU_MODEL_PATH, 
        custom_objects={'cosine_loss': cosine_loss, 'L2NormLayer': L2NormLayer}
    )
    
    # Előre kiszámolt CNN vektorok betöltése
    with open(CNN_DICT_PATH, 'rb') as f:
        cnn_vectors_dict = pickle.load(f)

    print("\n2. Teszt-lejátszási listák betöltése és szűrése...")
    test_pids = np.load(TEST_PIDS_PATH)
    test_pids_set = set(test_pids)
    playlist_map = defaultdict(list)
    
    with h5py.File(H5_PATH, "r") as hf:
        num_records = hf["playlist_tracks/pid"].shape[0]
        chunk_size = 5_000_000 # 5 millió soronként dolgozzuk fel
        
        for start in tqdm(range(0, num_records, chunk_size), desc="Adatok beolvasása"):
            end = min(start + chunk_size, num_records)
            pids_chunk = hf["playlist_tracks/pid"][start:end]
            uris_chunk = hf["playlist_tracks/track_uri"][start:end]
            
            for i in range(len(pids_chunk)):
                pid = pids_chunk[i]
                if pid in test_pids_set:
                    uri = uris_chunk[i].decode('utf-8') if isinstance(uris_chunk[i], bytes) else uris_chunk[i]
                    playlist_map[pid].append(uri)
    
    all_test_pids = list(playlist_map.keys())
    np.random.shuffle(all_test_pids)

    results = {"hr": [], "ndcg": [], "map": []}
    tested_count = 0
    
    print(f"\n3. GRU Hibrid kiértékelés indítása ({NUM_TEST_PLAYLISTS} lista)...")
    pbar = tqdm(total=NUM_TEST_PLAYLISTS)

    for pid in all_test_pids:
        if tested_count >= NUM_TEST_PLAYLISTS: break
        
        songs = playlist_map[pid]
        if len(songs) < 3: continue 
        
        # Keressük az utolsó lehetséges horgonyt (CNN dal)
        anchor_idx = -1
        for i in range(len(songs) - 2, -1, -1):
            if songs[i] in cnn_vectors_dict:
                anchor_idx = i
                break
        
        if anchor_idx == -1: continue 
        
        # Előzmények (Context) és a jósolandó dalok (Target)
        context_uris = songs[max(0, anchor_idx + 1 - SEQ_LEN) : anchor_idx + 1]
        target_uris = set(songs[anchor_idx + 1:]) 
        
        # Bemeneti szekvencia összeállítása (Hibrid mód)
        seq_vectors = []
        for i, uri in enumerate(context_uris):
            if i == len(context_uris) - 1: # Az utolsó dal kapja a CNN vektort
                seq_vectors.append(cnn_vectors_dict[uri])
            else: # A többi marad Word2Vec
                if uri in w2v_model.wv:
                    seq_vectors.append(w2v_model.wv[uri])
                else:
                    seq_vectors.append(np.zeros(400))
        
        # GRU predikció
        X_input = pad_sequences([seq_vectors], maxlen=SEQ_LEN, dtype='float32', padding='pre')
        predicted_vector = gru_model.predict(X_input, verbose=0)[0]
        
        full_history_set = set(songs[:anchor_idx + 1])
        # Legközelebbi szomszédok keresése (K+len, hogy ki tudjuk szűrni az ismerteket)
        sims = w2v_model.wv.most_similar(positive=[predicted_vector], topn=K_VALUE + len(full_history_set))
        
        # Ajánlási lista szűrése (ne ajánljuk azt, ami már benne van a listában)
        recs = [u for u, s in sims if u not in full_history_set][:K_VALUE]
        
        # Metrikák számítása
        hr, ndcg, ap = calculate_ranking_metrics(recs, target_uris, K_VALUE)
        results["hr"].append(hr)
        results["ndcg"].append(ndcg)
        results["map"].append(ap)
        
        tested_count += 1
        pbar.update(1)

    pbar.close()

    # --- EREDMÉNYEK ÖSSZESÍTÉSE ---
    print("\n" + "="*60)
    print(f"🏆 HIBRID GRU4REC KIÉRTÉKELÉS (K={K_VALUE}, Teszt halmazon)")
    print("="*60)
    print(f"Sikeresen tesztelt listák: {tested_count}")
    print("-" * 60)
    print(f"Hit Rate@{K_VALUE}:  {np.mean(results['hr'])*100:.2f}%")
    print(f"NDCG@{K_VALUE}:      {np.mean(results['ndcg']):.4f}")
    print(f"MAP@{K_VALUE}:       {np.mean(results['map']):.4f}")
    print("="*60)

if __name__ == "__main__":
    main()

1. Modellek és CNN szótár betöltése...

2. Teszt-lejátszási listák betöltése és szűrése...


Adatok beolvasása: 100%|██████████| 14/14 [00:39<00:00,  2.85s/it]



3. GRU Hibrid kiértékelés indítása (1000 lista)...


100%|██████████| 1000/1000 [01:45<00:00,  9.50it/s]



🏆 HIBRID GRU4REC KIÉRTÉKELÉS (K=20, Teszt halmazon)
Sikeresen tesztelt listák: 1000
------------------------------------------------------------
Hit Rate@20:  2.90%
NDCG@20:      0.0075
MAP@20:       0.0080


In [4]:
import os
import pickle
import h5py
import numpy as np
import tensorflow as tf
from gensim.models import Word2Vec
from collections import defaultdict
from tqdm import tqdm
from tensorflow.keras.preprocessing.sequence import pad_sequences

# ==========================================
# --- ÚTVONALAK ÉS BEÁLLÍTÁSOK ---
# ==========================================
H5_PATH = "../Dataset/spotify_dataset_compressed.h5"
W2V_PATH = "../Models/song2vec.model"
# IDE ÍRD BE AZ ÉPPEN TESZTELNI KÍVÁNT EPOCH MODELLJÉT (pl. 32 vagy 40):
GRU_MODEL_PATH = "../Models/spotify_gru4rec_hybrid_ep32.keras"
CNN_DICT_PATH = "../Models/cnn_vectors.pkl" 
TEST_PIDS_PATH = "../test_pids.npy"

K_VALUE = 20 
NUM_TEST_PLAYLISTS = 1000 
SEQ_LEN = 10 

# ==========================================
# 1. CUSTOM RÉTEGEK ÉS METRIKÁK
# ==========================================
class L2NormLayer(tf.keras.layers.Layer):
    def call(self, x): 
        return tf.math.l2_normalize(x, axis=1)

def cosine_loss(y_true, y_pred):
    return 1.0 - tf.reduce_mean(tf.reduce_sum(tf.math.l2_normalize(y_true, axis=1) * y_pred, axis=1))

def calculate_ranking_metrics(recommended_uris, relevant_uris, K):
    """HR, NDCG és AP számítása a Top-K ajánlások alapján."""
    top_k = recommended_uris[:K]
    hits = [1 if uri in relevant_uris else 0 for uri in top_k]
    
    # Hit Rate
    hr = 1 if sum(hits) > 0 else 0
    
    # Average Precision (MAP-hez)
    precs = []
    num_hits = 0
    for i, hit in enumerate(hits):
        if hit:
            num_hits += 1
            precs.append(num_hits / (i + 1))
    ap = np.mean(precs) if precs else 0
    
    # NDCG
    dcg = 0
    for i, hit in enumerate(hits):
        if hit:
            dcg += 1 / np.log2(i + 2)
    
    idcg = 0
    for i in range(min(len(relevant_uris), K)):
        idcg += 1 / np.log2(i + 2)
    
    ndcg = dcg / idcg if idcg > 0 else 0
    
    return hr, ndcg, ap

# ==========================================
# 2. KIÉRTÉKELŐ FŐ CIKLUS
# ==========================================
def main():
    print("1. Modellek és CNN szótár betöltése...")
    # Word2Vec betöltése mmap módban (RAM spórolás)
    w2v_model = Word2Vec.load(W2V_PATH, mmap='r')
    
    # GRU Hibrid modell betöltése a custom objektumokkal
    gru_model = tf.keras.models.load_model(
        GRU_MODEL_PATH, 
        custom_objects={'cosine_loss': cosine_loss, 'L2NormLayer': L2NormLayer}
    )
    
    # Előre kiszámolt CNN vektorok betöltése
    with open(CNN_DICT_PATH, 'rb') as f:
        cnn_vectors_dict = pickle.load(f)

    print("\n2. Teszt-lejátszási listák betöltése és villámgyors szűrése...")
    # 1. Betöltjük az ELŐRE MEGKEVERT teszt azonosítókat a .npy fájlból
    all_test_pids = np.load(TEST_PIDS_PATH)
    
    # 2. AZONNAL levágjuk az első X darabot (Garantáltan ugyanaz az 1000 db vegyes lista lesz mindig)
    selected_test_pids = all_test_pids[:NUM_TEST_PLAYLISTS]
    
    # 3. Halmazt csinálunk belőle a H5 szűréshez, hogy O(1) sebességű legyen a keresés
    test_pids_set = set(selected_test_pids) 
    
    playlist_map = defaultdict(list)
    
    # H5 fájl olvasása: Csak az 1000 kiválasztott listát engedjük a memóriába!
    with h5py.File(H5_PATH, "r") as hf:
        num_records = hf["playlist_tracks/pid"].shape[0]
        chunk_size = 5_000_000 # 5 millió soronként dolgozzuk fel
        
        for start in tqdm(range(0, num_records, chunk_size), desc="Adatok beolvasása (H5)"):
            end = min(start + chunk_size, num_records)
            pids_chunk = hf["playlist_tracks/pid"][start:end]
            uris_chunk = hf["playlist_tracks/track_uri"][start:end]
            
            for i in range(len(pids_chunk)):
                pid = pids_chunk[i]
                if pid in test_pids_set: # <--- EZ A VILLÁMGYORS SZŰRÉS LELKE
                    uri = uris_chunk[i].decode('utf-8') if isinstance(uris_chunk[i], bytes) else uris_chunk[i]
                    playlist_map[pid].append(uri)

    results = {"hr": [], "ndcg": [], "map": []}
    tested_count = 0
    
    print(f"\n3. GRU Hibrid kiértékelés indítása ({NUM_TEST_PLAYLISTS} fix listán)...")
    pbar = tqdm(total=NUM_TEST_PLAYLISTS)

    # Végigmegyünk a fix 1000 listánkon
    for pid in selected_test_pids:
        # Biztonságos lekérés, hátha üres maradt a listában valami hiba folytán
        songs = playlist_map.get(pid, [])
        if len(songs) < 3: 
            pbar.update(1)
            continue 
        
        # Keressük az utolsó lehetséges horgonyt (CNN dal)
        anchor_idx = -1
        for i in range(len(songs) - 2, -1, -1):
            if songs[i] in cnn_vectors_dict:
                anchor_idx = i
                break
        
        if anchor_idx == -1: 
            pbar.update(1)
            continue 
        
        # Előzmények (Context) és a jósolandó dalok (Target)
        context_uris = songs[max(0, anchor_idx + 1 - SEQ_LEN) : anchor_idx + 1]
        target_uris = set(songs[anchor_idx + 1:]) 
        
        # Bemeneti szekvencia összeállítása (Hibrid mód)
        seq_vectors = []
        for i, uri in enumerate(context_uris):
            if i == len(context_uris) - 1: # Az utolsó dal kapja a CNN vektort
                seq_vectors.append(cnn_vectors_dict[uri])
            else: # A többi marad Word2Vec
                if uri in w2v_model.wv:
                    seq_vectors.append(w2v_model.wv[uri])
                else:
                    seq_vectors.append(np.zeros(400))
        
        # GRU predikció
        X_input = pad_sequences([seq_vectors], maxlen=SEQ_LEN, dtype='float32', padding='pre')
        predicted_vector = gru_model.predict(X_input, verbose=0)[0]
        
        # TELJES ELŐZMÉNY a szűréshez (Over-fetching pufferelés)
        full_history_set = set(songs[:anchor_idx + 1])
        
        # Legközelebbi szomszédok keresése (K+len, hogy ki tudjuk szűrni az ismerteket)
        sims = w2v_model.wv.most_similar(positive=[predicted_vector], topn=K_VALUE + len(full_history_set))
        
        # Ajánlási lista szűrése (ne ajánljuk azt, ami a teljes eddigi listában szerepelt)
        recs = [u for u, s in sims if u not in full_history_set][:K_VALUE]
        
        # Metrikák számítása
        hr, ndcg, ap = calculate_ranking_metrics(recs, target_uris, K_VALUE)
        results["hr"].append(hr)
        results["ndcg"].append(ndcg)
        results["map"].append(ap)
        
        tested_count += 1
        pbar.update(1)

    pbar.close()

    # --- EREDMÉNYEK ÖSSZESÍTÉSE ---
    print("\n" + "="*60)
    print(f"🏆 HIBRID GRU4REC KIÉRTÉKELÉS (K={K_VALUE}, Fixált Teszt Halmaz)")
    print("="*60)
    print(f"Sikeresen tesztelt listák: {tested_count}")
    print("-" * 60)
    if tested_count > 0:
        print(f"Hit Rate@{K_VALUE}:  {np.mean(results['hr'])*100:.2f}%")
        print(f"NDCG@{K_VALUE}:      {np.mean(results['ndcg']):.4f}")
        print(f"MAP@{K_VALUE}:       {np.mean(results['map']):.4f}")
    else:
        print("Nem volt elegendő megfelelő tesztadat a kiértékeléshez!")
    print("="*60)

if __name__ == "__main__":
    main()

1. Modellek és CNN szótár betöltése...

2. Teszt-lejátszási listák betöltése és villámgyors szűrése...


Adatok beolvasása (H5): 100%|██████████| 14/14 [00:30<00:00,  2.16s/it]



3. GRU Hibrid kiértékelés indítása (1000 fix listán)...


100%|██████████| 1000/1000 [01:46<00:00,  9.37it/s]



🏆 HIBRID GRU4REC KIÉRTÉKELÉS (K=20, Fixált Teszt Halmaz)
Sikeresen tesztelt listák: 967
------------------------------------------------------------
Hit Rate@20:  1.96%
NDCG@20:      0.0074
MAP@20:       0.0065


In [3]:
import os
import pickle
import h5py
import numpy as np
import tensorflow as tf
from gensim.models import Word2Vec
from collections import defaultdict
from tqdm import tqdm
from tensorflow.keras.preprocessing.sequence import pad_sequences

# ==========================================
# --- ÚTVONALAK ÉS BEÁLLÍTÁSOK ---
# ==========================================
H5_PATH = "../Dataset/spotify_dataset_compressed.h5"
W2V_PATH = "../Models/song2vec.model"
# IDE ÍRD BE AZ ÉPPEN TESZTELNI KÍVÁNT EPOCH MODELLJÉT (pl. 32 vagy 40):
GRU_MODEL_PATH = "../Models/spotify_gru4rec_hybrid_ep40.keras"
CNN_DICT_PATH = "../Models/cnn_vectors.pkl" 
TEST_PIDS_PATH = "../test_pids.npy"

K_VALUE = 20 
NUM_TEST_PLAYLISTS = 1000 
SEQ_LEN = 10 

# ==========================================
# 1. CUSTOM RÉTEGEK ÉS METRIKÁK
# ==========================================
class L2NormLayer(tf.keras.layers.Layer):
    def call(self, x): 
        return tf.math.l2_normalize(x, axis=1)

def cosine_loss(y_true, y_pred):
    return 1.0 - tf.reduce_mean(tf.reduce_sum(tf.math.l2_normalize(y_true, axis=1) * y_pred, axis=1))

def calculate_ranking_metrics(recommended_uris, relevant_uris, K):
    """HR, NDCG és AP számítása a Top-K ajánlások alapján."""
    top_k = recommended_uris[:K]
    hits = [1 if uri in relevant_uris else 0 for uri in top_k]
    
    # Hit Rate
    hr = 1 if sum(hits) > 0 else 0
    
    # Average Precision (MAP-hez)
    precs = []
    num_hits = 0
    for i, hit in enumerate(hits):
        if hit:
            num_hits += 1
            precs.append(num_hits / (i + 1))
    ap = np.mean(precs) if precs else 0
    
    # NDCG
    dcg = 0
    for i, hit in enumerate(hits):
        if hit:
            dcg += 1 / np.log2(i + 2)
    
    idcg = 0
    for i in range(min(len(relevant_uris), K)):
        idcg += 1 / np.log2(i + 2)
    
    ndcg = dcg / idcg if idcg > 0 else 0
    
    return hr, ndcg, ap

# ==========================================
# 2. KIÉRTÉKELŐ FŐ CIKLUS
# ==========================================
def main():
    print("1. Modellek és CNN szótár betöltése...")
    # Word2Vec betöltése mmap módban (RAM spórolás)
    w2v_model = Word2Vec.load(W2V_PATH, mmap='r')
    
    # GRU Hibrid modell betöltése a custom objektumokkal
    gru_model = tf.keras.models.load_model(
        GRU_MODEL_PATH, 
        custom_objects={'cosine_loss': cosine_loss, 'L2NormLayer': L2NormLayer}
    )
    
    # Előre kiszámolt CNN vektorok betöltése
    with open(CNN_DICT_PATH, 'rb') as f:
        cnn_vectors_dict = pickle.load(f)

    print("\n2. Teszt-lejátszási listák betöltése és villámgyors szűrése...")
    # 1. Betöltjük az ELŐRE MEGKEVERT teszt azonosítókat a .npy fájlból
    all_test_pids = np.load(TEST_PIDS_PATH)
    
    # 2. AZONNAL levágjuk az első X darabot (Garantáltan ugyanaz az 1000 db vegyes lista lesz mindig)
    selected_test_pids = all_test_pids[:NUM_TEST_PLAYLISTS]
    
    # 3. Halmazt csinálunk belőle a H5 szűréshez, hogy O(1) sebességű legyen a keresés
    test_pids_set = set(selected_test_pids) 
    
    playlist_map = defaultdict(list)
    
    # H5 fájl olvasása: Csak az 1000 kiválasztott listát engedjük a memóriába!
    with h5py.File(H5_PATH, "r") as hf:
        num_records = hf["playlist_tracks/pid"].shape[0]
        chunk_size = 5_000_000 # 5 millió soronként dolgozzuk fel
        
        for start in tqdm(range(0, num_records, chunk_size), desc="Adatok beolvasása (H5)"):
            end = min(start + chunk_size, num_records)
            pids_chunk = hf["playlist_tracks/pid"][start:end]
            uris_chunk = hf["playlist_tracks/track_uri"][start:end]
            
            for i in range(len(pids_chunk)):
                pid = pids_chunk[i]
                if pid in test_pids_set: # <--- EZ A VILLÁMGYORS SZŰRÉS LELKE
                    uri = uris_chunk[i].decode('utf-8') if isinstance(uris_chunk[i], bytes) else uris_chunk[i]
                    playlist_map[pid].append(uri)

    results = {"hr": [], "ndcg": [], "map": []}
    tested_count = 0
    
    print(f"\n3. GRU Hibrid kiértékelés indítása ({NUM_TEST_PLAYLISTS} fix listán)...")
    pbar = tqdm(total=NUM_TEST_PLAYLISTS)

    # Végigmegyünk a fix 1000 listánkon
    for pid in selected_test_pids:
        # Biztonságos lekérés, hátha üres maradt a listában valami hiba folytán
        songs = playlist_map.get(pid, [])
        if len(songs) < 3: 
            pbar.update(1)
            continue 
        
        # Keressük az utolsó lehetséges horgonyt (CNN dal)
        anchor_idx = -1
        for i in range(len(songs) - 2, -1, -1):
            if songs[i] in cnn_vectors_dict:
                anchor_idx = i
                break
        
        if anchor_idx == -1: 
            pbar.update(1)
            continue 
        
        # Előzmények (Context) és a jósolandó dalok (Target)
        context_uris = songs[max(0, anchor_idx + 1 - SEQ_LEN) : anchor_idx + 1]
        target_uris = set(songs[anchor_idx + 1:]) 
        
        # Bemeneti szekvencia összeállítása (Hibrid mód)
        seq_vectors = []
        for i, uri in enumerate(context_uris):
            if i == len(context_uris) - 1: # Az utolsó dal kapja a CNN vektort
                seq_vectors.append(cnn_vectors_dict[uri])
            else: # A többi marad Word2Vec
                if uri in w2v_model.wv:
                    seq_vectors.append(w2v_model.wv[uri])
                else:
                    seq_vectors.append(np.zeros(400))
        
        # GRU predikció
        X_input = pad_sequences([seq_vectors], maxlen=SEQ_LEN, dtype='float32', padding='pre')
        predicted_vector = gru_model.predict(X_input, verbose=0)[0]
        
        # TELJES ELŐZMÉNY a szűréshez (Over-fetching pufferelés)
        full_history_set = set(songs[:anchor_idx + 1])
        
        # Legközelebbi szomszédok keresése (K+len, hogy ki tudjuk szűrni az ismerteket)
        sims = w2v_model.wv.most_similar(positive=[predicted_vector], topn=K_VALUE + len(full_history_set))
        
        # Ajánlási lista szűrése (ne ajánljuk azt, ami a teljes eddigi listában szerepelt)
        recs = [u for u, s in sims if u not in full_history_set][:K_VALUE]
        
        # Metrikák számítása
        hr, ndcg, ap = calculate_ranking_metrics(recs, target_uris, K_VALUE)
        results["hr"].append(hr)
        results["ndcg"].append(ndcg)
        results["map"].append(ap)
        
        tested_count += 1
        pbar.update(1)

    pbar.close()

    # --- EREDMÉNYEK ÖSSZESÍTÉSE ---
    print("\n" + "="*60)
    print(f"🏆 HIBRID GRU4REC KIÉRTÉKELÉS (K={K_VALUE}, Fixált Teszt Halmaz)")
    print("="*60)
    print(f"Sikeresen tesztelt listák: {tested_count}")
    print("-" * 60)
    if tested_count > 0:
        print(f"Hit Rate@{K_VALUE}:  {np.mean(results['hr'])*100:.2f}%")
        print(f"NDCG@{K_VALUE}:      {np.mean(results['ndcg']):.4f}")
        print(f"MAP@{K_VALUE}:       {np.mean(results['map']):.4f}")
    else:
        print("Nem volt elegendő megfelelő tesztadat a kiértékeléshez!")
    print("="*60)

if __name__ == "__main__":
    main()

1. Modellek és CNN szótár betöltése...

2. Teszt-lejátszási listák betöltése és villámgyors szűrése...


Adatok beolvasása (H5): 100%|██████████| 14/14 [00:33<00:00,  2.40s/it]



3. GRU Hibrid kiértékelés indítása (1000 fix listán)...


100%|██████████| 1000/1000 [01:25<00:00, 11.67it/s]



🏆 HIBRID GRU4REC KIÉRTÉKELÉS (K=20, Fixált Teszt Halmaz)
Sikeresen tesztelt listák: 967
------------------------------------------------------------
Hit Rate@20:  1.76%
NDCG@20:      0.0074
MAP@20:       0.0072


In [5]:
import os
import pickle
import h5py
import numpy as np
import tensorflow as tf
from gensim.models import Word2Vec
from collections import defaultdict
from tqdm import tqdm
from tensorflow.keras.preprocessing.sequence import pad_sequences

# ==========================================
# --- ÚTVONALAK ÉS BEÁLLÍTÁSOK ---
# ==========================================
H5_PATH = "../Dataset/spotify_dataset_compressed.h5"
W2V_PATH = "../Models/song2vec.model"
# IDE ÍRD BE AZ ÉPPEN TESZTELNI KÍVÁNT EPOCH MODELLJÉT (pl. 32 vagy 40):
GRU_MODEL_PATH = "../Models/spotify_gru4rec_hybrid_ep50.keras"
CNN_DICT_PATH = "../Models/cnn_vectors.pkl" 
TEST_PIDS_PATH = "../test_pids.npy"

K_VALUE = 20 
NUM_TEST_PLAYLISTS = 1000 
SEQ_LEN = 10 

# ==========================================
# 1. CUSTOM RÉTEGEK ÉS METRIKÁK
# ==========================================
class L2NormLayer(tf.keras.layers.Layer):
    def call(self, x): 
        return tf.math.l2_normalize(x, axis=1)

def cosine_loss(y_true, y_pred):
    return 1.0 - tf.reduce_mean(tf.reduce_sum(tf.math.l2_normalize(y_true, axis=1) * y_pred, axis=1))

def calculate_ranking_metrics(recommended_uris, relevant_uris, K):
    """HR, NDCG és AP számítása a Top-K ajánlások alapján."""
    top_k = recommended_uris[:K]
    hits = [1 if uri in relevant_uris else 0 for uri in top_k]
    
    # Hit Rate
    hr = 1 if sum(hits) > 0 else 0
    
    # Average Precision (MAP-hez)
    precs = []
    num_hits = 0
    for i, hit in enumerate(hits):
        if hit:
            num_hits += 1
            precs.append(num_hits / (i + 1))
    ap = np.mean(precs) if precs else 0
    
    # NDCG
    dcg = 0
    for i, hit in enumerate(hits):
        if hit:
            dcg += 1 / np.log2(i + 2)
    
    idcg = 0
    for i in range(min(len(relevant_uris), K)):
        idcg += 1 / np.log2(i + 2)
    
    ndcg = dcg / idcg if idcg > 0 else 0
    
    return hr, ndcg, ap

# ==========================================
# 2. KIÉRTÉKELŐ FŐ CIKLUS
# ==========================================
def main():
    print("1. Modellek és CNN szótár betöltése...")
    # Word2Vec betöltése mmap módban (RAM spórolás)
    w2v_model = Word2Vec.load(W2V_PATH, mmap='r')
    
    # GRU Hibrid modell betöltése a custom objektumokkal
    gru_model = tf.keras.models.load_model(
        GRU_MODEL_PATH, 
        custom_objects={'cosine_loss': cosine_loss, 'L2NormLayer': L2NormLayer}
    )
    
    # Előre kiszámolt CNN vektorok betöltése
    with open(CNN_DICT_PATH, 'rb') as f:
        cnn_vectors_dict = pickle.load(f)

    print("\n2. Teszt-lejátszási listák betöltése és villámgyors szűrése...")
    # 1. Betöltjük az ELŐRE MEGKEVERT teszt azonosítókat a .npy fájlból
    all_test_pids = np.load(TEST_PIDS_PATH)
    
    # 2. AZONNAL levágjuk az első X darabot (Garantáltan ugyanaz az 1000 db vegyes lista lesz mindig)
    selected_test_pids = all_test_pids[:NUM_TEST_PLAYLISTS]
    
    # 3. Halmazt csinálunk belőle a H5 szűréshez, hogy O(1) sebességű legyen a keresés
    test_pids_set = set(selected_test_pids) 
    
    playlist_map = defaultdict(list)
    
    # H5 fájl olvasása: Csak az 1000 kiválasztott listát engedjük a memóriába!
    with h5py.File(H5_PATH, "r") as hf:
        num_records = hf["playlist_tracks/pid"].shape[0]
        chunk_size = 5_000_000 # 5 millió soronként dolgozzuk fel
        
        for start in tqdm(range(0, num_records, chunk_size), desc="Adatok beolvasása (H5)"):
            end = min(start + chunk_size, num_records)
            pids_chunk = hf["playlist_tracks/pid"][start:end]
            uris_chunk = hf["playlist_tracks/track_uri"][start:end]
            
            for i in range(len(pids_chunk)):
                pid = pids_chunk[i]
                if pid in test_pids_set: # <--- EZ A VILLÁMGYORS SZŰRÉS LELKE
                    uri = uris_chunk[i].decode('utf-8') if isinstance(uris_chunk[i], bytes) else uris_chunk[i]
                    playlist_map[pid].append(uri)

    results = {"hr": [], "ndcg": [], "map": []}
    tested_count = 0
    
    print(f"\n3. GRU Hibrid kiértékelés indítása ({NUM_TEST_PLAYLISTS} fix listán)...")
    pbar = tqdm(total=NUM_TEST_PLAYLISTS)

    # Végigmegyünk a fix 1000 listánkon
    for pid in selected_test_pids:
        # Biztonságos lekérés, hátha üres maradt a listában valami hiba folytán
        songs = playlist_map.get(pid, [])
        if len(songs) < 3: 
            pbar.update(1)
            continue 
        
        # Keressük az utolsó lehetséges horgonyt (CNN dal)
        anchor_idx = -1
        for i in range(len(songs) - 2, -1, -1):
            if songs[i] in cnn_vectors_dict:
                anchor_idx = i
                break
        
        if anchor_idx == -1: 
            pbar.update(1)
            continue 
        
        # Előzmények (Context) és a jósolandó dalok (Target)
        context_uris = songs[max(0, anchor_idx + 1 - SEQ_LEN) : anchor_idx + 1]
        target_uris = set(songs[anchor_idx + 1:]) 
        
        # Bemeneti szekvencia összeállítása (Hibrid mód)
        seq_vectors = []
        for i, uri in enumerate(context_uris):
            if i == len(context_uris) - 1: # Az utolsó dal kapja a CNN vektort
                seq_vectors.append(cnn_vectors_dict[uri])
            else: # A többi marad Word2Vec
                if uri in w2v_model.wv:
                    seq_vectors.append(w2v_model.wv[uri])
                else:
                    seq_vectors.append(np.zeros(400))
        
        # GRU predikció
        X_input = pad_sequences([seq_vectors], maxlen=SEQ_LEN, dtype='float32', padding='pre')
        predicted_vector = gru_model.predict(X_input, verbose=0)[0]
        
        # TELJES ELŐZMÉNY a szűréshez (Over-fetching pufferelés)
        full_history_set = set(songs[:anchor_idx + 1])
        
        # Legközelebbi szomszédok keresése (K+len, hogy ki tudjuk szűrni az ismerteket)
        sims = w2v_model.wv.most_similar(positive=[predicted_vector], topn=K_VALUE + len(full_history_set))
        
        # Ajánlási lista szűrése (ne ajánljuk azt, ami a teljes eddigi listában szerepelt)
        recs = [u for u, s in sims if u not in full_history_set][:K_VALUE]
        
        # Metrikák számítása
        hr, ndcg, ap = calculate_ranking_metrics(recs, target_uris, K_VALUE)
        results["hr"].append(hr)
        results["ndcg"].append(ndcg)
        results["map"].append(ap)
        
        tested_count += 1
        pbar.update(1)

    pbar.close()

    # --- EREDMÉNYEK ÖSSZESÍTÉSE ---
    print("\n" + "="*60)
    print(f"🏆 HIBRID GRU4REC KIÉRTÉKELÉS (K={K_VALUE}, Fixált Teszt Halmaz)")
    print("="*60)
    print(f"Sikeresen tesztelt listák: {tested_count}")
    print("-" * 60)
    if tested_count > 0:
        print(f"Hit Rate@{K_VALUE}:  {np.mean(results['hr'])*100:.2f}%")
        print(f"NDCG@{K_VALUE}:      {np.mean(results['ndcg']):.4f}")
        print(f"MAP@{K_VALUE}:       {np.mean(results['map']):.4f}")
    else:
        print("Nem volt elegendő megfelelő tesztadat a kiértékeléshez!")
    print("="*60)

if __name__ == "__main__":
    main()

1. Modellek és CNN szótár betöltése...

2. Teszt-lejátszási listák betöltése és villámgyors szűrése...


Adatok beolvasása (H5): 100%|██████████| 14/14 [00:36<00:00,  2.64s/it]



3. GRU Hibrid kiértékelés indítása (1000 fix listán)...


100%|██████████| 1000/1000 [01:45<00:00,  9.46it/s]



🏆 HIBRID GRU4REC KIÉRTÉKELÉS (K=20, Fixált Teszt Halmaz)
Sikeresen tesztelt listák: 967
------------------------------------------------------------
Hit Rate@20:  1.86%
NDCG@20:      0.0077
MAP@20:       0.0078
